# Implementing a web search tool

## Preparations

Sign up at https://www.tavily.com, get an API key and add it to the environment. Then, install `tavily-pytion` client.

## Verify basic web searching function using Tavily

We verify that we can search the web using the `TavilyClient`:


In [1]:
import os
from tavily import TavilyClient

tavily_client = TavilyClient()

In [2]:
def search_web(query: str, max_results: int = 2) -> list:
  response = tavily_client.search(query, max_results=max_results)
  return response.get("results")

Lets search for Kipchoge's marathon record to verify search_web.

In [3]:
search_web("Kipcoge's marathon world record")

[{'url': 'https://en.wikipedia.org/wiki/Eliud_Kipchoge',
  'title': 'Eliud Kipchoge',
  'content': 'On 16 September, Kipchoge won the 2018 Berlin Marathon in a time of 2:01:39, breaking the previous world record by 1 minute and 18 seconds (2:02:57 set by fellow countryman Dennis Kimetto at the Berlin Marathon in 2014). On 20 January, Kipchoge announced his desire to win all six World Marathon Majors (he had already won three, the London, Berlin, and Chicago marathons by that time). On 25 September, Kipchoge won the Berlin Marathon decisively in a time of 2:01:09, beating by 30 seconds his own previous world record, which he set on the same course in 2018. "Berlin Marathon Results: Eliud Kipchoge Breaks World Record". **^** "Eliud Kipchoge smashes world marathon record by 78 seconds in Berlin". **^** "The Greatest Ever – 2:01:39 – Eliud Kipchoge Crushes World Record to Win 2018 Berlin Marathon". "Eliud Kipchoge Crushes Marathon World Record at Berlin Marathon". **^** "Eliud Kipchoge bre

### Adding search options

We expand our search function with several options:

- `topic` specifies a content type like: general, news or finance.
- `time_range` filters results by recency, when we need current vs historical information
- `country` prefers search results from a specific country

See https://docs.tavily.com/documentation/api-reference/endpoint/search for more information.

> The more options a tool exposes, the more complex your tool definitions becomes, and the harder it is for the LLM to use it correctly.


In [4]:
def search_web(
    query: str,
    max_results: int = 2,
    topic: str = "general",
    time_range: str | None = None,
    country: str | None = None
) -> list:
  
  response = tavily_client.search(
    query,
    max_results=max_results,
    topic=topic,
    time_range=time_range,
    country=country
  )
  return response.get("results")

### Handle errors gracefully

In the following example, we handle errors with a catch-all approach. In production, different errors require different handling:

- 401 Authentication errors indicate configuration problems
- 429 Rate limit errors benefit from retry with backoff
- Timeout errors might need a simpler query
- Network errors are different from API errors

In [10]:
def search_web(
    query: str,
    max_results: int = 2,
    topic: str = "general",
    time_range: str | None = None,
    country: str | None = None
) -> list | str:
    """Search the web for the given query."""
  
    try:
        response = tavily_client.search(
            query,
            max_results=max_results,
            topic=topic,
            time_range=time_range,
            country=country
        )
        return response.get("results")
    
    except Exception as e:
        return f"Error: Search failed - {e}"


### Converting to tool definitions

To use our tools in combination with an LLM, we need to define them in a standardized tool definition format. Functions may change frequently, so manual updating these definitions is error-prone and inefficient. So we generate these definitions automatically by converting Python functions into a tool definition.

We use the `inspect`module to extract:
- functions name
- docstring
- parameter details

In [11]:
import inspect
 
def example_tool(input_1:str, input_2:int=1):
    """docstring for example_tool"""
    return
        
print(f"function name: {example_tool.__name__}")
print(f"function docstring: {example_tool.__doc__}")
print(f"function signature: {inspect.signature(example_tool)}")

function name: example_tool
function docstring: docstring for example_tool
function signature: (input_1: str, input_2: int = 1)


In [12]:
def function_to_input_schema(func) -> dict:
    type_map = {
        str: "string",
        int: "integer",
        float: "number",
        bool: "boolean",
        list: "array",
        dict: "object",
        type(None): "null",
    }
    
    try:
        signature = inspect.signature(func)
    except ValueError as e:
        raise ValueError(
            f"Failed to get signature for function {func.__name__}: {str(e)}"
        )
    
    parameters = {}
    for param in signature.parameters.values():
        try:
            param_type = type_map.get(param.annotation, "string")
        except KeyError as e:
            raise KeyError(
                f"Unknown type annotation {param.annotation} for parameter {param.name}: {str(e)}"
            )
        parameters[param.name] = {"type": param_type}
    
    required = [
        param.name
        for param in signature.parameters.values()
        if param.default == inspect._empty
    ]
    
    return {
            "type": "object",
            "properties": parameters,
            "required": required,
        }

The signature is used to extract function parameters and convert their types. If a parameter has no default value, it is required. Lets try to convert our web search tool:

In [ ]:
def format_tool_definition(name: str, description: str, parameters: dict) -> dict:
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": parameters,
        },
    }
 
def function_to_tool_definition(func) -> dict:
    return format_tool_definition(
        func.__name__,
        func.__doc__ or "",
        function_to_input_schema(func)
    )
 
search_tool_definition = function_to_tool_definition(search_web)
print(search_tool_definition)